# Module 5 — Inverse Kinematics
## Unit 8 — Mini Project: Reach the Fruit
### Lesson 8.1 — The Project: From Target Pose to Joint Angles

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Demonstration (§7)
The integrated reach_the_fruit entry point on the worked example.


In [ ]:
import numpy as np

def fk_two_link(t1, t2, L1, L2):
    return np.array([L1*np.cos(t1)+L2*np.cos(t1+t2), L1*np.sin(t1)+L2*np.sin(t1+t2)])

def jacobian_2link(t1, t2, L1, L2):
    s1,s12=np.sin(t1),np.sin(t1+t2); c1,c12=np.cos(t1),np.cos(t1+t2)
    return np.array([[-L1*s1-L2*s12,-L2*s12],[L1*c1+L2*c12,L2*c12]])

def ik_2link_closed(x, y, L1, L2):
    c2=(x*x+y*y-L1*L1-L2*L2)/(2*L1*L2)
    if c2<-1-1e-9 or c2>1+1e-9: return []
    c2=max(-1.0,min(1.0,c2)); sols=[]
    for sign in (+1.0,-1.0):
        s2=sign*np.sqrt(max(0.0,1-c2*c2)); t2=np.arctan2(s2,c2)
        t1=np.arctan2(y,x)-np.arctan2(L2*np.sin(t2),L1+L2*np.cos(t2))
        sols.append((t1,t2))
        if abs(s2)<1e-6: break
    return sols

def reachable(x, y, L1, L2, tol=1e-9):
    r=np.hypot(x,y); return abs(L1-L2)-tol<=r<=L1+L2+tol

def ik_numerical(target, theta0, L1, L2, alpha=1.0, tol=1e-6, max_iter=100):
    theta=np.array(theta0,float); target=np.array(target,float)
    for it in range(max_iter):
        e=target-fk_two_link(theta[0],theta[1],L1,L2)
        if np.linalg.norm(e)<tol: return theta
        J=jacobian_2link(theta[0],theta[1],L1,L2)
        theta=theta+alpha*np.linalg.pinv(J)@e
    return theta

def norm180(a): return (a+180.0)%360.0-180.0

T=np.array([[1,0,0.1],[0,1,0.0],[0,0,1.0]])
def feasible(s, limits):
    d1,d2=norm180(np.degrees(s[0])),norm180(np.degrees(s[1]))
    return limits[0][0]<=d1<=limits[0][1] and limits[1][0]<=d2<=limits[1][1]

def reach_the_fruit(p_cam, T_base_cam, L1, L2, limits, theta_cur, tol=1e-3):
    p_base=(T_base_cam@np.array([p_cam[0],p_cam[1],1.0]))[:2]
    if not reachable(p_base[0],p_base[1],L1,L2): return None,"unreachable"
    cands=ik_2link_closed(p_base[0],p_base[1],L1,L2)
    if not cands: return None,"no_solution"
    verified=[c for c in cands if np.linalg.norm(p_base-fk_two_link(c[0],c[1],L1,L2))<tol]
    if not verified: return None,"verification_failed"
    feas=[c for c in verified if feasible(c,limits)]
    if not feas: return None,"no_feasible"
    best=min(feas,key=lambda s:np.hypot(norm180(np.degrees(s[0])-np.degrees(theta_cur[0])),
                                        norm180(np.degrees(s[1])-np.degrees(theta_cur[1]))))
    return best,"ok"

L1,L2=0.4,0.3
res,st=reach_the_fruit((0.4,0.2),T,L1,L2,[(-45,45),(0,150)],np.radians([-30,80]))
print("Reach the Fruit:", None if res is None else np.round(np.degrees(res),1), "status:", st)


## Coding Exercise (§8)
The integrated call returns a verified, feasible configuration.


In [ ]:
# YOUR CODE HERE
res,st=reach_the_fruit((0.4,0.2),T,L1,L2,[(-45,45),(0,150)],np.radians([-30,80]))
assert st=="ok"
# verified: FK lands on the base-frame target
p_base=(T@np.array([0.4,0.2,1.0]))[:2]
assert np.allclose(fk_two_link(res[0],res[1],L1,L2), p_base, atol=1e-3)
# feasible: within limits
assert -45<=np.degrees(res[0])<=45 and 0<=np.degrees(res[1])<=150
print("capstone entry point OK")


## Check your work


In [ ]:
res,st=reach_the_fruit((0.4,0.2),T,L1,L2,[(-45,45),(0,150)],np.radians([-30,80]))
assert st=="ok" and res is not None
print("All checks passed.")
